# Moonshine Bengali ASR — Inference
Runs inference on your test set using the fine-tuned checkpoint.

In [ ]:
# ── Install deps (skip if already installed in your Kaggle env) ──
!pip install -q transformers librosa soundfile jiwer

In [ ]:
import os, torch, librosa, numpy as np, pandas as pd
from pathlib import Path
from transformers import AutoProcessor, MoonshineForConditionalGeneration

# ── Paths (from your Kaggle dataset structure) ───────────────────
MODEL_PATH  = "/kaggle/input/upgrade-to-python-3-11/moonshine-bengali-tiny/final"
AUDIO_DIR   = "/kaggle/input/datasets/ahmfuad01/refined-test/final_16k"
OUTPUT_CSV  = "/kaggle/working/predictions.csv"

SAMPLE_RATE     = 16_000
MAX_SEC         = 25.0
CHUNK_SEC       = 20.0
OVERLAP_SEC     = 1.5
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")
print(f"Model  : {MODEL_PATH}")
print(f"Audio  : {AUDIO_DIR}")

In [ ]:
# ── Load model ───────────────────────────────────────────────────
processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = MoonshineForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)
model.eval()
print("Model loaded ✅")

In [ ]:
# ── Inference helpers ─────────────────────────────────────────────

def _transcribe_chunk(chunk):
    inputs = processor(chunk, sampling_rate=SAMPLE_RATE, return_tensors="pt")
    iv = inputs["input_values"].to(DEVICE)
    if DEVICE == "cuda":
        iv = iv.half()
    with torch.no_grad():
        ids = model.generate(iv, max_new_tokens=448)
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip()


def transcribe(path):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    duration  = len(audio) / SAMPLE_RATE

    if duration <= MAX_SEC:
        return _transcribe_chunk(audio)

    # Long audio → chunk-and-stitch
    chunk_samples   = int(CHUNK_SEC   * SAMPLE_RATE)
    overlap_samples = int(OVERLAP_SEC * SAMPLE_RATE)
    step_samples    = chunk_samples - overlap_samples

    parts, start = [], 0
    while start < len(audio):
        end = min(start + chunk_samples, len(audio))
        parts.append(_transcribe_chunk(audio[start:end]))
        if end == len(audio): break
        start += step_samples

    return " ".join(parts)

In [ ]:
# ── Run batch inference ───────────────────────────────────────────
files = sorted(Path(AUDIO_DIR).glob("*.wav"))
print(f"Found {len(files)} test files\n")

records = []
for i, fp in enumerate(files, 1):
    text = transcribe(str(fp))
    records.append({"filename": fp.name, "transcript": text})
    print(f"[{i:>2}/{len(files)}] {fp.name}  →  {text}")

df = pd.DataFrame(records)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"\n✅ Saved to {OUTPUT_CSV}")
df

In [ ]:
# ── (Optional) WER evaluation if you have ground-truth labels ────
# Fill in your reference transcripts here, one per file in the same order.
#
# from jiwer import wer
# references   = ["রেফারেন্স ট্রান্সক্রিপ্ট ১", "রেফারেন্স ট্রান্সক্রিপ্ট ২", ...]
# hypotheses   = df["transcript"].tolist()
# print(f"WER: {wer(references, hypotheses):.2%}")